In [1]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import multiprocessing
import torch
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import ConcatDataset,DataLoader, Dataset, Subset 
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import v2
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchinfo import summary
from tqdm.auto import tqdm
from PIL import Image
import random
import time
import copy
import timm 

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
# Check accelerator

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [3]:
# Set SEED for reproducibility

SEED = 24520152

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'
TRAIN_DIR

'/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [5]:
INPUT_DIR = '/kaggle/input/mobilenetv2-5fcv-hieu-42'
INPUT_DIR

'/kaggle/input/mobilenetv2-5fcv-hieu-42'

In [6]:
SAVE_DIR = '/kaggle/working/'
SAVE_DIR

'/kaggle/working/'

In [7]:
# 3 labels: Glioma Tumor, Meningioma Tumor, Pituitary Tumor

CLASS_NAMES = sorted([d for d in os.listdir(os.path.join(TRAIN_DIR, 'Subset_1')) if os.path.isdir(os.path.join(TRAIN_DIR, 'Subset_1', d))])
CLASS_NAMES

['Glioma Tumor', 'Meningioma Tumor', 'Pituitary Tumor']

In [8]:
# Model name
MODEL_NAME = 'MobileNetV2'
MODEL_NAME

'MobileNetV2'

In [9]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3
DROPOUT_RATE = 0.3

In [10]:
def get_data_for_fold(k: int, train_dir: str = TRAIN_DIR, batch_size: int = BATCH_SIZE) -> tuple[DataLoader, DataLoader]:
    """
    Creates PyTorch DataLoaders for K-Fold Cross-Validation, specifically tailored
    for MobileNetV2 transfer learning with ImageNet standards.

    This function sets up a data pipeline that:
    1. Applies standard ImageNet normalization statistics.
    2. Implements strong data augmentation for the training set (using torchvision v2).
    3. Prepares validation data with deterministic preprocessing (Resize -> CenterCrop).
    4. Handles subset selection based on the current fold index 'k'.

    Args:
        k (int): The index of the validation fold (e.g., 1 to 5). 
                 The folder 'Subset_{k}' will be used for validation, while 
                 all other subsets are concatenated for training.
        train_dir (str): Path to the root directory containing the fold subsets 
                         (e.g., 'Subset_1', 'Subset_2', etc.).
        batch_size (int, optional): Number of samples per batch. Defaults to 32.

    Returns:
        tuple: A tuple containing:
            - train_loader (DataLoader): The training data loader (shuffled).
            - val_loader (DataLoader): The validation data loader (not shuffled).
    """

    # Standard ImageNet normalization statistics (Required for MobileNetV2 weights)
    norm_mean=[0.485, 0.456, 0.406]
    norm_std=[0.229, 0.224, 0.225]

    # Training Transform Pipeline
    train_transform = v2.Compose([
        # Resize to 256x256 first. This provides a buffer for subsequent 
        # rotation/translation and cropping, preventing black border artifacts.
        v2.Resize(size=256),

        # Apply Data Augmentation
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(degrees=36),
        v2.RandomAffine(degrees=0, scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),

        # Use CenterCrop to focus on the primary subject
        v2.CenterCrop(size=224),

        # Convert PIL/Numpy to Tensor, cast to Float32, and rescale to [0, 1]
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),

        # Normalize using ImageNet mean and std
        v2.Normalize(mean=norm_mean, std=norm_std),
    ])

    # Validation Transform Pipeline
    val_transform = v2.Compose([
        v2.Resize(size=256),
        v2.CenterCrop(size=224),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])

    # Data Path Setup
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    train_dirs = [os.path.join(train_dir, f'Subset_{i}') for i in range(1, 6) if i != k]

    # Worker Configuration
    # Determine the optimal number of CPU workers to prevent bottlenecks.
    # Capped at 4 to avoid excessive memory overhead.
    num_workers = min(4, os.cpu_count())

    # Validation Loader
    val_dataset = ImageFolder(root=val_dir, transform=val_transform)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    # Training Loader
    list_train_dataset = [ImageFolder(root=td, transform=train_transform) for td in train_dirs]
    train_dataset = ConcatDataset(datasets=list_train_dataset)
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader

In [11]:
DATASET_CACHE = {}

def get_valid_data_for_fold(k: int, train_dir: str = TRAIN_DIR, batch_size: int = BATCH_SIZE) -> DataLoader:
    """
    Retrieves the validation DataLoader for a specific fold with caching mechanism.
    This function checks if the validation loader for the specified fold `k` is already
    present in the global `DATASET_CACHE`. If found, it returns the cached loader to save memory 
    and processing time. If not, it generates the loader using `get_data_for_fold`, caches it, 
    and then returns it.

    Args:
        k (int): The fold index (e.g., 1 to 5) identifying which subset is used for validation.
        train_dir (str, optional): The root directory path of the training data. 
            Defaults to global TRAIN_DIR.
        batch_size (int, optional): The number of samples per batch to load. 
            Defaults to global BATCH_SIZE.

    Returns:
        DataLoader: The PyTorch DataLoader containing the validation dataset for fold `k`.
    """
    
    # 1. Check if the loader for this fold is already in the cache
    if k in DATASET_CACHE:
        return DATASET_CACHE[k]
    
    # 2. If not in cache, create new loaders
    # Only need val_loader
    _, val_loader = get_data_for_fold(k=k, train_dir=train_dir, batch_size=batch_size)
    
    # 3. Store the newly created validation loader in the cache
    DATASET_CACHE[k] = val_loader

    return val_loader

In [12]:
class MobileNetV2(nn.Module):
    """
    MobileNetV2-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: MobileNetV2 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> None:
        super().__init__()

        # Load Pre-trained MobileNetV2
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        backbone = models.mobilenet_v2(weights=weights)

        # MobileNetV2 .features contains all the convolutional layers (Inverted Residuals)
        self.features = backbone.features

        # Freezing parameters to prevent updating during training
        for param in self.features.parameters():
            param.requires_grad = False

        # Define Custom Classifier Head
        # MobileNetV2 output feature map has 1280 channels
        self.in_features = 1280 
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1280, H, W) -> (Batch, 1280, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1280, 1, 1) -> (Batch, 1280)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """

        # Feature extraction (Frozen)
        x = self.features(x)

        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)

        # Output Logits
        logits = self.classifier(x)

        return logits

In [13]:
def build_mobilenetv2(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> MobileNetV2:
    """
    Factory function to instantiate the customized MobileNetV2 model for Transfer Learning.

    This function initializes a `MobileNetV2` which includes:
    1. A frozen MobileNetV2 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        MobileNetV2: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = MobileNetV2(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [14]:
def get_model_for_fold(i: int, k: int, input_dir: str = INPUT_DIR)-> nn.Module:
    """
    Loads the trained MobileNetV2 model weights for a specific block and fold.

    Args:
        i (int): Block index (e.g., 6, 5, 4...).
        k (int): Fold index (1-5).
        input_dir (str): Directory containing .pth files.
    Returns:
        nn.Module: The model with loaded weights set to eval mode, or None if file missing.
    """
   
    # Construct model
    model = build_mobilenetv2().to(device)
    
    # Make file path
    # f'{model_name}_block_{block}_fold_{k}.pth'
    filename = f'{MODEL_NAME}_block_{i}_fold_{k}.pth'
    filepath = os.path.join(input_dir, filename)
    
    # Load weights
    if os.path.exists(filepath):
        # Load state_dict 
        state_dict = torch.load(filepath, map_location=device)
        model.load_state_dict(state_dict)
        print(f"Loaded: {filename}")
    else:
        print(f"Warning: File {filename} not found at {input_dir}")
        return None

    # Evaluation mode
    model.to(device)
    model.eval() 
    
    return model

In [15]:
def get_predictions_for_fold(k: int, input_dir: str = INPUT_DIR) -> tuple[np.ndarray, np.ndarray]:
    """
    Retrieves true labels and predictions for all unfreezing blocks in a specific fold.
    This function iterates through all saved checkpoints (blocks 1 to 6) for a given fold,
    loads each model, performs inference on the validation set, and aggregates the results.
    Args:
        k: The current fold index (e.g., 1 to 5).
        input_dir: Directory path where (.pth files) are saved.
    Returns:
        tuple: A tuple containing:
            - y_true: The ground truth labels. Shape: (num_samples,)
            - y_pred: Predictions from all blocks. 
              Shape: where 6 corresponds to blocks 1-6.
    """
    
    # valid loader
    valid_loader = get_valid_data_for_fold(k)

    # Extract ground truth labels
    y_true = []
    for _, labels in valid_loader:
        y_true.extend(labels.numpy())
    y_true = np.array(y_true)

    # Collect Predictions for each Block (y_pred)
    y_pred_all_blocks = []

    # Loop through blocks 1 to 5 (Mobile Net)
    for i in range(1, 6):
        # Load the model using the helper function
        model = get_model_for_fold(i, k, input_dir)
        
        if model is None:
            # Handle missing files if necessary (fill with zeros or skip)
            print(f"Skipping block {i} - fold {k} due to missing file.")
            continue

        # Loop
        preds_block = []
        
        with torch.no_grad(): 
            for inputs, _ in valid_loader:
                inputs = inputs.to(device)
                
                # Forward pass
                outputs = model(inputs)
                
                # Apply Softmax to convert logits to probabilities
                probs = torch.softmax(outputs, dim=1)
                preds_block.extend(probs.cpu().numpy())
        # Save y_pred
        y_pred_all_blocks.append(np.array(preds_block))

        # Memory Cleanup
        del model
        torch.cuda.empty_cache()
        gc.collect()

    # Convert list of arrays to a single numpy array
    # shape: (num_blocks, num_samples, num_classes)
    y_pred = np.array(y_pred_all_blocks)
    
    return y_true, y_pred

In [16]:
def calculate_macro_specificity(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 3) -> float:
    """
    Computes the Macro-Average Specificity (True Negative Rate) for multi-class classification.

    This function calculates specificity using the "One-vs-Rest" strategy:
    1. For each class, it treats that class as "Positive" and all other classes as "Negative".
    2. It computes True Negatives (TN) and False Positives (FP) for that specific class.
    3. It calculates specificity for that class using the formula: Specificity = TN / (TN + FP).
    4. Finally, it returns the unweighted mean (macro-average) of specificity scores across all classes.

    Args:
        y_true (np.ndarray): 1D array containing the ground truth class labels. 
            Shape: (n_samples,). Example: [0, 1, 2, 0]
        y_pred (np.ndarray): 1D array containing the predicted class labels (not probabilities). 
            Shape: (n_samples,). Example: [0, 2, 2, 0]
        num_classes (int, optional): The total number of unique classes in the dataset. 
            Defaults to 3.

    Returns:
        float: The macro-averaged specificity score. 
               Range is [0.0, 1.0], where 1.0 indicates perfect identification of negative cases.

    Note:
        A small epsilon (1e-8) is added to the denominator to prevent ZeroDivisionError 
        in cases where (TN + FP) equals 0.
    """
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Initialize list to store specificity for each class
    specs = []

    for i in range(num_classes):
        # True Positives
        TP = cm[i, i]
        # False Negatives:
        FN = cm[i, :].sum() - TP
        # False Positives
        FP = cm[:, i].sum() - TP
        # True Negatives
        TN = cm.sum() - (TP + FN + FP)
        # Specificity
        specificity = TN / (TN + FP + 1e-8)
        specs.append(specificity)

    # Return the mean to get a single Macro Average score
    return np.array(specs)

In [17]:
def evaluate_fold(k: int, input_dir: str = INPUT_DIR) -> np.ndarray:
    """
    Evaluates the model performance for a specific fold across all unfreezing blocks and
    computes evaluation metrics (Acc, Prec, Rec, F1, Spec).
    Args:
        k: Fold index (1-5).
        input_dir: Path to saved models.
    Returns:
        np.ndarray: A matrix of metrics for all blocks.
                    Shape: (num_blocks, 5_metrics).
                    Columns order: [Recall, Specificity, Precision, F1-Score, Accuracy].
    """
    # Get prediction
    y_true, y_pred = get_predictions_for_fold(k, input_dir)

    model_results = []
    
    # Loop through each block's prediction
    for i in range(len(y_pred)):
        # prevent model predict is vector zeros
        if np.all(y_pred[i] == 0):
            model_results.append([0,0,0,0,0])
            continue

        # Convert Probabilities 
        y_pred_label = np.argmax(y_pred[i], axis=1)

        # Metrics Calculation
        acc = accuracy_score(y_true, y_pred_label)
        # Macro Average for multi-class
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        # Specificity 
        spec_array = calculate_macro_specificity(y_true, y_pred_label, num_classes=3)
        specificity = np.mean(spec_array) # Take Mean to get one single score
        # Order: Recall, Specificity, Precision, F1, Accuracy
        metrics_row = [recall, specificity, precision, f1, acc]
        model_results.append(metrics_row)

    return np.array(model_results)

In [18]:
five_fold_results = []
# main loop, from fold 1 - 5
for i in range(1, 6):
    five_fold_results.append(evaluate_fold(i, INPUT_DIR))

five_fold_results = np.array(five_fold_results)

num_fold = 5
num_model = 5
metric_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for idx, metric_name in enumerate(metric_names):
        # Take metrics of all folds at model(m), at column metric (idx)
        values = five_fold_results[:, m, idx]
        avg_metrics[m][metric_name] = values.mean() * 100

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 134MB/s]


Loaded: MobileNetV2_block_1_fold_1.pth
Loaded: MobileNetV2_block_2_fold_1.pth
Loaded: MobileNetV2_block_3_fold_1.pth
Loaded: MobileNetV2_block_4_fold_1.pth
Loaded: MobileNetV2_block_5_fold_1.pth
Loaded: MobileNetV2_block_1_fold_2.pth
Loaded: MobileNetV2_block_2_fold_2.pth
Loaded: MobileNetV2_block_3_fold_2.pth
Loaded: MobileNetV2_block_4_fold_2.pth
Loaded: MobileNetV2_block_5_fold_2.pth
Loaded: MobileNetV2_block_1_fold_3.pth
Loaded: MobileNetV2_block_2_fold_3.pth
Loaded: MobileNetV2_block_3_fold_3.pth
Loaded: MobileNetV2_block_4_fold_3.pth
Loaded: MobileNetV2_block_5_fold_3.pth
Loaded: MobileNetV2_block_1_fold_4.pth
Loaded: MobileNetV2_block_2_fold_4.pth
Loaded: MobileNetV2_block_3_fold_4.pth
Loaded: MobileNetV2_block_4_fold_4.pth
Loaded: MobileNetV2_block_5_fold_4.pth
Loaded: MobileNetV2_block_1_fold_5.pth
Loaded: MobileNetV2_block_2_fold_5.pth
Loaded: MobileNetV2_block_3_fold_5.pth
Loaded: MobileNetV2_block_4_fold_5.pth
Loaded: MobileNetV2_block_5_fold_5.pth


In [19]:
row_names = ['FT: B$_1$-B$_5$', 'FT: B$_2$-B$_5$', 'FT: B$_3$-B$_5$', 'FT: B$_4$-B$_5$', 'FT: B$_5$']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result.index.name = 'Fine-tuning'
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result = df_result.reset_index()
df_result.style.hide(axis='index').format(precision=2)

Fine-tuning,Recall,Specificity,Precision,F1-Score,Accuracy
FT: B$_1$-B$_5$,92.74,96.65,92.19,92.39,93.21
FT: B$_2$-B$_5$,92.25,96.46,92.12,92.15,93.02
FT: B$_3$-B$_5$,90.58,95.71,89.97,90.18,91.27
FT: B$_4$-B$_5$,89.57,95.21,89.13,89.25,90.36
FT: B$_5$,83.90,92.73,84.33,84.00,85.91


In [20]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{llrrrrr}
\toprule
 & Fine-tuning & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
0 & FT: B$_1$-B$_5$ & 92.74 & 96.65 & 92.19 & 92.39 & 93.21 \\
1 & FT: B$_2$-B$_5$ & 92.25 & 96.46 & 92.12 & 92.15 & 93.02 \\
2 & FT: B$_3$-B$_5$ & 90.58 & 95.71 & 89.97 & 90.18 & 91.27 \\
3 & FT: B$_4$-B$_5$ & 89.57 & 95.21 & 89.13 & 89.25 & 90.36 \\
4 & FT: B$_5$ & 83.90 & 92.73 & 84.33 & 84.00 & 85.91 \\
\bottomrule
\end{tabular}
\end{table}

